# Fetch population density raster (geoTIFF)



In [1]:
# Imports and general variables

from owslib.wms import WebMapService
import rasterio as rio
import requests, sys, re, os
from urllib.parse import urlencode
from pathlib import Path
from tqdm import tqdm

In [15]:
# Parameters

WMS_URL = "https://geodata.scb.se/geoserver/stat/wms"
GRID_RES = "1km"
YEAR = "2024"
LAYER_NAME = f"befolkning_{GRID_RES}_{YEAR}"  # <-- fixed name
OUTFILE = f"SCB_tot_pop_{YEAR}_{GRID_RES}.tif"

In [12]:
# Functions

def download(url, out_path):
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(out_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))

In [16]:
# Connect to SCB WMS

print('Connecting to SCB WMS …')
wms = WebMapService(WMS_URL, version='1.3.0')

md = wms[LAYER_NAME]
print('✔ Layer found:', LAYER_NAME)

Connecting to SCB WMS …
✔ Layer found: befolkning_1km_2024


In [17]:
bbox_data = next((b for b in md.boundingBox if b[-1]=='EPSG:3006'), None)
crs, bbox = ('EPSG:3006', bbox_data[:4]) if bbox_data else ('EPSG:4326', md.boundingBoxWGS84)

TypeError: 'float' object is not subscriptable

In [9]:
bbox_data = next((b for b in md.boundingBox if b[-1]=='EPSG:3006'), None)
crs, bbox = ('EPSG:3006', bbox_data[:4]) if bbox_data else ('EPSG:4326', md.boundingBoxWGS84)

params = dict(
    service='WMS', version='1.3.0', request='GetMap',
    layers=layer, styles='',
    crs=crs, bbox=",".join(map(str, bbox)),
    width=4096, height=4096,
    format='image/geotiff', transparent='false'
)
url = f"{WMS_URL}?{urlencode(params)}"

print('Downloading GeoTIFF …')
download(url, OUTFILE)

with rio.open(OUTFILE) as src:
    print('Saved:', OUTFILE, '|', src.crs, src.width, '×', src.height)

Connecting to SCB WMS …


ValueError: No layer matching year=2023, grid=1km